In [0]:
from pyspark.sql import functions as F
import uuid

CATALOG     = "claims"
AUDIT_TABLE = f"{CATALOG}.gold.dq_audit"

RUN_ID, _results = str(uuid.uuid4()), []

def check(layer, table, name, passed, observed, expected, severity="critical"):
    passed = bool(passed)
    _results.append((RUN_ID, layer, table, name, severity,
                     passed, str(observed), str(expected)))
    print(f"[{'PASS' if passed else 'FAIL'}] {table}.{name}  "
          f"observed={observed}  expected={expected}")

def flush():
    if not _results:
        return
    schema = ("run_id string, layer string, table_name string, check_name string, "
              "severity string, passed boolean, observed string, expected string")
    (spark.createDataFrame(_results, schema)
        .withColumn("checked_at", F.current_timestamp())
        .write.mode("append").saveAsTable(AUDIT_TABLE))
    print(f"wrote {len(_results)} results")

print("ready, run_id =", RUN_ID)

In [0]:
proc = spark.table("claims.gold.agg_procedure_payment")

print("=== codes failing payment_rate range ===")
(proc.filter("payment_rate > 100 OR payment_rate < 0 OR payment_rate IS NULL")
     .select("hcpcs_code","line_count","total_allowed","total_paid","payment_rate")
     .orderBy(F.desc("total_paid"))
     .show(30, False))

print("=== how many have zero or null allowed? ===")
(proc.select(
    F.sum(F.when(F.col("total_allowed").isNull(), 1).otherwise(0)).alias("null_allowed"),
    F.sum(F.when(F.col("total_allowed") == 0, 1).otherwise(0)).alias("zero_allowed"),
    F.sum(F.when(F.col("total_paid") > F.col("total_allowed"), 1).otherwise(0)).alias("paid_gt_allowed"))
 .show(1, False))

print("=== month series ===")
(spark.table("claims.gold.agg_monthly_claims")
     .select("service_month").distinct().orderBy("service_month")
     .show(50, False))

In [0]:
_results.clear()

lines = spark.table("claims.silver.service_line")
hdr   = spark.table("claims.silver.claim_header")
proc  = spark.table("claims.gold.agg_procedure_payment")
prov  = spark.table("claims.gold.agg_provider_payment")
mon   = spark.table("claims.gold.agg_monthly_claims")

# --- grain ---
t, u = proc.count(), proc.select("hcpcs_code").distinct().count()
check("gold", "agg_procedure_payment", "grain_one_row_per_hcpcs", t == u, u, t)

pt = prov.count()
pu = prov.select("provider_id", "claim_type").distinct().count()
check("gold", "agg_provider_payment", "grain_provider_claimtype", pt == pu, pu, pt)

# --- reconciliation to silver ---
silver_paid = lines.filter("hcpcs_code IS NOT NULL") \
                   .agg(F.sum("line_paid")).first()[0] or 0
gold_paid   = proc.agg(F.sum("total_paid")).first()[0] or 0
check("gold", "agg_procedure_payment", "paid_reconciles_to_silver",
      gold_paid <= silver_paid, f"{gold_paid:,.0f}", f"<= {silver_paid:,.0f}")

hdr_claims = hdr.count()
mon_claims = mon.agg(F.sum("claims")).first()[0] or 0
check("gold", "agg_monthly_claims", "claim_count_reconciles",
      hdr_claims == mon_claims, mon_claims, hdr_claims)

# --- payment rate ---
uncomputable = proc.filter(
    F.col("total_allowed").isNull() | (F.col("total_allowed") <= 0)).count()
check("gold", "agg_procedure_payment", "allowed_amount_populated",
      uncomputable == 0, uncomputable, 0, severity="warn")

# sanity bound, not a hard ceiling — overpayment is real, not a computation error
absurd = proc.filter(
    (F.col("total_allowed") > 0) & ~F.col("payment_rate").between(0, 200)).count()
check("gold", "agg_procedure_payment", "payment_rate_sane", absurd == 0, absurd, 0)

# overpayment tracked as a business finding
overpaid = proc.filter(F.col("total_paid") > F.col("total_allowed")).count()
check("gold", "agg_procedure_payment", "no_overpayment",
      overpaid == 0, overpaid, 0, severity="warn")

# --- other metric bounds ---
bad_zero = proc.filter(~F.col("zero_pay_pct").between(0, 100)).count()
check("gold", "agg_procedure_payment", "zero_pay_pct_in_range", bad_zero == 0, bad_zero, 0)

mismatch = proc.filter(
    F.abs(F.col("total_allowed") - F.col("total_paid") - F.col("total_variance")) > 1).count()
check("gold", "agg_procedure_payment", "variance_identity_holds", mismatch == 0, mismatch, 0)

neg = proc.filter((F.col("total_paid") < 0) | (F.col("total_allowed") < 0)).count()
check("gold", "agg_procedure_payment", "amounts_non_negative", neg == 0, neg, 0)

below = proc.filter("line_count < 100").count()
check("gold", "agg_procedure_payment", "volume_filter_applied", below == 0, below, 0)

# --- monthly series ---
months = mon.select("service_month").distinct().count()
span   = mon.agg(F.min("service_month"), F.max("service_month")).first()
expected_months = (span[1].year - span[0].year) * 12 + (span[1].month - span[0].month) + 1
check("gold", "agg_monthly_claims", "no_month_gaps",
      months == expected_months, months, expected_months, severity="warn")

future = mon.filter(F.col("service_month") > F.lit("2011-01-01")).count()
check("gold", "agg_monthly_claims", "no_future_months", future == 0, future, 0, severity="warn")

flush()

failed = [r for r in _results if not r[5] and r[4] == "critical"]
if failed:
    raise Exception(f"{len(failed)} critical gold check(s) failed: {[r[3] for r in failed]}")
print("all critical gold checks passed")

In [0]:
%sql
SELECT run_id, COUNT(*) FROM claims.gold.dq_audit GROUP BY run_id ORDER BY 2 DESC;

In [0]:
_results.clear()

In [0]:
%sql
DELETE FROM claims.gold.dq_audit
WHERE run_id = '0fb8a5e5-bd5d-4136-97e7-4665144da7be';